# ️ Glu-Stock: 04_BACKTEST_LAB
**Phase**: Historical Validation (2025 Simulation) | v18.22 (Total Recall)

This notebook simulates the v18 ensemble with Weighted Logic (40% LGBM / 60% CNN) and Mid-Term Trend Filter (SMA 50) over 2025 data.

In [ ]:
#!pip install -q yfinance pandas scikit-learn joblib tensorflow ta lightgbm


In [ ]:
import os, json, joblib, numpy as np, pandas as pd, yfinance as yf, warnings, ta
try: import tensorflow.lite as tflite
except: import tflite_runtime.interpreter as tflite
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

def find_model_file(filename):
    search_paths = ['/kaggle/input', '/kaggle/working', '.', 'data/models']
    for root_dir in search_paths:
        if not os.path.exists(root_dir): continue
        for root, dirs, files in os.walk(root_dir):
            if filename in files: return os.path.join(root, filename)
    return None

def frac_diff(series, d=0.4, window=100):
    w = [1.0]
    for k in range(1, window):
        w.append(-w[-1] * (d - k + 1) / k)
    w = np.array(w[::-1])
    result = np.full(len(series), np.nan)
    for t in range(window - 1, len(series)):
        result[t] = np.dot(w, series[t - window + 1:t + 1])
    return result

class MLPredictor:
    def __init__(self, path):
        brain = joblib.load(path)
        self.model = brain.get('model')
        self.features = brain.get('features', [])

    def build_features(self, df):
        close = df['Close'].squeeze()
        high = df['High'].squeeze()
        low = df['Low'].squeeze()
        volume = df['Volume'].squeeze()
        feat = pd.DataFrame(index=df.index)
        feat['Returns'] = close.pct_change()
        feat['RSI'] = ta.momentum.RSIIndicator(close=close, window=14).rsi()
        macd = ta.trend.MACD(close=close)
        feat['MACD'] = macd.macd_diff()
        boll = ta.volatility.BollingerBands(close=close, window=20, window_dev=2)
        feat['BB_High'] = boll.bollinger_hband_indicator()
        feat['BB_Low'] = boll.bollinger_lband_indicator()
        feat['ATR'] = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()
        feat['ADX'] = ta.trend.ADXIndicator(high=high, low=low, close=close, window=14).adx()
        obv = ta.volume.OnBalanceVolumeIndicator(close=close, volume=volume).on_balance_volume()
        feat['OBV_norm'] = (obv - obv.rolling(20).mean()) / (obv.rolling(20).std() + 1e-7)
        feat['day_of_week'] = df.index.dayofweek
        feat['week_of_month'] = (df.index.day - 1) // 7
        feat['frac_diff_close'] = frac_diff(close.values.flatten(), d=0.4, window=100)
        vol_ma = volume.rolling(20).mean()
        feat['vol_ratio'] = volume / (vol_ma + 1e-7)
        return feat.dropna()

    def predict(self, df):
        try:
            feat = self.build_features(df)
            if len(feat) == 0: return 0.0
            row = feat[self.features].tail(1)
            proba = self.model.predict_proba(row)[0]
            return float(proba[1])
        except: return 0.0

class CNNPredictor:
    def __init__(self, path):
        self.interpreter = tflite.Interpreter(model_path=path)
        self.interpreter.allocate_tensors()

    def predict(self, df):
        try:
            close = df['Close'].squeeze().values[-30:]
            high = df['High'].squeeze().values[-30:]
            low = df['Low'].squeeze().values[-30:]
            volume = df['Volume'].squeeze().values[-30:]
            opn = df['Open'].squeeze().values[-30:]
            raw = np.column_stack([opn, high, low, close, volume])
            seq_min, seq_max = raw.min(axis=0), raw.max(axis=0)
            norm_seq = (raw - seq_min) / (seq_max - seq_min + 1e-7)
            input_details = self.interpreter.get_input_details()
            input_data = np.expand_dims(norm_seq.astype(np.float32), axis=0)
            self.interpreter.set_tensor(input_details[0]['index'], input_data)
            self.interpreter.invoke()
            output = self.interpreter.get_tensor(self.interpreter.get_output_details()[0]['index'])[0]
            return float(output[1]) if len(output) > 1 else float(output[0])
        except: return 0.5


In [ ]:
def run_total_recall_validation():
    print("Starting Total Recall Validation (Weighted v18.22)...\n")
    
    lgbm_path = find_model_file('glu_brain_v1.joblib')
    cnn_path = find_model_file('cnn_daily_t2.tflite')
    
    if not lgbm_path or not cnn_path: return
    
    lgbm = MLPredictor(lgbm_path)
    cnn = CNNPredictor(cnn_path)
    
    cohort = ['BBCA.JK', 'TLKM.JK', 'ASII.JK', 'ADRO.JK', 'BMRI.JK', 'BBRI.JK', 'ICBP.JK', 'PTBA.JK', 'ANTM.JK', 'UNTR.JK']
    data = yf.download(cohort + ['^JKSE'], start='2024-01-01', end='2025-12-31', progress=False, auto_adjust=True)
    
    valid_dates = [d for d in data.index if d.year == 2025]
    print(f"Executing simulation over {len(valid_dates)} trading days...")
    signals_found = []
    
    for date in valid_dates[::2]: # High-resolution verification
        for ticker in cohort:
            try:
                df = data.loc[:date, (slice(None), ticker)]
                df.columns = df.columns.droplevel(1)
                df = df.dropna()
                
                if len(df) < 150: continue
                
                # S1: SMA 50 TREND Guard (Mid-Term)
                sma50 = df['Close'].tail(50).mean()
                if df['Close'].iloc[-1] < sma50: continue
                
                # S2: WEIGHTED Ensemble (Total Recall)
                l_prob = lgbm.predict(df)
                c_prob = cnn.predict(df)
                
                ensemble_score = (l_prob * 0.4) + (c_prob * 0.6)
                
                if ensemble_score >= 0.5:
                    signals_found.append({
                        'date': date.strftime('%Y-%m-%d'),
                        'ticker': ticker,
                        'L_Prob': f"{l_prob:.2%}",
                        'C_Prob': f"{c_prob:.2%}",
                        'Final_Score': f"{ensemble_score:.2%}"
                    })
            except: continue
            
    results = pd.DataFrame(signals_found)
    print(f"\n[SUCCESS] Simulation Complete.")
    print(f"Total Signals Found in 2025: {len(results)}")
    
    if not results.empty:
        print("\n--- Restored Signal Log (Top 40) ---")
        print(results.head(40).to_string(index=False))
    else:
        print("\n[FAIL] Still zero signals. Please verify if models were correctly loaded.")

run_total_recall_validation()
